# The LangGraph Supervisor Agent

Lab 4 Part A gave you a Genie space that answers questions about sensor readings.
Lab 2 gave you a graph of aircraft, systems, components, flights, and maintenance
history. Lab 3 gave you the maintenance manuals, chunked and embedded, with
retrievers that walk out from a matched passage into the graph around it.

Three tools, three question types, and no single tool that answers a real
question. Here you build the thing that decides which one to reach for.

**Prerequisites**

| Lab | What this notebook needs from it |
|---|---|
| [Lab 2](../Lab_2_Databricks_ETL_Neo4j) | The fleet graph in your own Aura instance |
| [Lab 3 notebook 01](../Lab_3_Semantic_Search/01_data_and_embeddings.ipynb) | The `fleet-ops-<your-user>` secret scope, and the `maintenanceChunkEmbeddings` vector index |
| [Lab 3 notebook 02](../Lab_3_Semantic_Search/02_graphrag_retrievers.ipynb) | The `VectorCypherRetriever` this notebook's manual tool is built from |
| [Lab 4 Part A](../Lab_4_Compound_AI_Agents/PART_A.md) | Your Genie space, and its space ID |

**Learning objectives**

- Build three tools that answer from three different stores
- Write a supervisor prompt that separates them, including the two that look alike
- Wire them into a LangGraph `StateGraph` with a routing loop
- Run one question that needs all three, and read the route it took
- Measure routing accuracy, and measure the hard pair on its own

## The shape of the agent

```
                    question
                       |
                       v
              +------------------+
              |    supervisor    |<---------+
              | Llama 3.3 70B    |          |
              +------------------+          |
                       |                    |
        +--------------+--------------+     |
        v              v              v     |
  +-----------+  +-----------+  +----------------+
  | genie     |  | cypher    |  | graphrag       |
  | Delta     |  | Neo4j     |  | Neo4j vector   |
  | telemetry |  | traversal |  | + Cypher tail  |
  +-----------+  +-----------+  +----------------+
        |              |              |     |
        +--------------+--------------+-----+
                       |
                       v
                  synthesize --> answer
```

Each tool reports back to the supervisor, which either asks for another tool or
stops. That loop is what lets one question use three tools in sequence, with each
result informing the next choice.

The interesting part is not the wiring. It is the prompt. `cypher_node` and
`graphrag_node` both end in a graph traversal, and a model that decides on the
ending cannot tell them apart. Section 6 is about that.

## Section 1: Configuration

Two things to set. Your Genie space ID, which you noted at the end of Lab 4
Part A, and your Neo4j database name.

Everything else is read from the secret scope Lab 3 notebook 01 created. The
scope name is derived from `current_user()`, so it is the same name in both
notebooks with nothing to copy across.

In [ ]:
# ==================================================
# CONFIGURATION - replace GENIE_SPACE_ID with yours
# ==================================================

# From Lab 4 Part A. Open your Genie space and take the ID out of the URL:
#   https://<workspace>/genie/rooms/<GENIE_SPACE_ID>
GENIE_SPACE_ID = "01f1661b55731a0293c3f84ac9c5ba52"

NEO4J_DATABASE = "neo4j"  # Neo4j database to use (Aura default is "neo4j")

import sys

sys.path.insert(0, ".")

from tools import secret_scope_name

SECRET_SCOPE = secret_scope_name(spark)
print(f"Secret scope:   {SECRET_SCOPE}")
print(f"Genie space ID: {GENIE_SPACE_ID}")

### If you skipped Lab 3

The cell below creates the secret scope and writes your Aura credentials into
it. It is the same block as Lab 3 notebook 01, kept here as a recovery path.

Run it only if the next cell tells you the scope is missing. It is commented out
because the normal path is that Lab 3 already did this, and rewriting a scope you
already have is a way to end up with two sets of credentials and one working
instance.

Note that `graphrag_node` also needs the vector index Lab 3 notebook 01 builds,
and this cell does not create that. Running this block gets you two tools out of
three. The agent still runs, and Section 5 shows you what it says when the manual
tool is missing.

In [ ]:
# ==================================================
# RECOVERY ONLY - run this if you skipped Lab 3
# ==================================================
#
# from databricks.sdk import WorkspaceClient
# from databricks.sdk.errors import ResourceAlreadyExists
#
# from data_utils import (
#     SECRET_KEY_NEO4J_PASSWORD,
#     SECRET_KEY_NEO4J_URI,
#     SECRET_KEY_NEO4J_USERNAME,
# )
#
# NEO4J_URI = "neo4j+s://xxxxxxxx.databases.neo4j.io"  # your Aura URI
# NEO4J_USERNAME = "neo4j"
# NEO4J_PASSWORD = ""  # your Aura password
#
# w = WorkspaceClient()
# try:
#     w.secrets.create_scope(scope=SECRET_SCOPE)
#     print(f"Created scope: {SECRET_SCOPE}")
# except ResourceAlreadyExists:
#     print(f"Scope already exists, reusing it: {SECRET_SCOPE}")
#
# w.secrets.put_secret(
#     scope=SECRET_SCOPE, key=SECRET_KEY_NEO4J_URI, string_value=NEO4J_URI.strip()
# )
# w.secrets.put_secret(
#     scope=SECRET_SCOPE,
#     key=SECRET_KEY_NEO4J_USERNAME,
#     string_value=NEO4J_USERNAME.strip(),
# )
# w.secrets.put_secret(
#     scope=SECRET_SCOPE, key=SECRET_KEY_NEO4J_PASSWORD, string_value=NEO4J_PASSWORD
# )
# print("Credentials stored. Clear this cell's output before saving the notebook.")

## Section 2: Connections

`tools.py` sits next to this notebook and holds the node builders, the prompts,
and the graph schema. It imports the embedder and the LLM from Lab 3's
`data_utils.py` rather than defining its own, which matters more than it looks:
the vectors in your `maintenanceChunkEmbeddings` index were written by that
embedder, and a query embedded by a different model does not match them.

In [ ]:
from databricks.sdk import WorkspaceClient

from tools import (
    EMBEDDING_MODEL,
    SUPERVISOR_MODEL,
    VECTOR_INDEX_NAME,
    build_cypher_node,
    build_genie_node,
    build_graphrag_node,
    build_supervisor_node,
    build_synthesize_node,
    get_embedder,
    get_llm,
    open_driver_from_secrets,
    vector_index_exists,
)

# The password is read, used, and dropped inside open_driver_from_secrets, so it
# never lands in a notebook variable.
driver = open_driver_from_secrets(dbutils, SECRET_SCOPE)

llm = get_llm(SUPERVISOR_MODEL)
embedder = get_embedder(EMBEDDING_MODEL)
workspace = WorkspaceClient()

print(f"Neo4j:      connected to database '{NEO4J_DATABASE}'")
print(f"Supervisor: {SUPERVISOR_MODEL}")
print(f"Embedder:   {EMBEDDING_MODEL}")

In [ ]:
# What is actually in your graph. The counts come from your own Aura instance.
records, _, _ = driver.execute_query(
    """
    CALL () { MATCH (n) RETURN count(n) AS nodes }
    CALL () { MATCH ()-[r]->() RETURN count(r) AS rels }
    CALL () { MATCH (c:Chunk) RETURN count(c) AS chunks }
    RETURN nodes, rels, chunks
    """,
    database_=NEO4J_DATABASE,
)
stats = records[0]
print(f"Nodes:         {stats['nodes']:,}")
print(f"Relationships: {stats['rels']:,}")
print(f"Manual chunks: {stats['chunks']:,}")
print(
    f"Vector index '{VECTOR_INDEX_NAME}': "
    f"{'online' if vector_index_exists(driver, VECTOR_INDEX_NAME, NEO4J_DATABASE) else 'MISSING'}"
)

## Section 3: `genie_node`, the telemetry tool

Your Genie space is already an agent. It takes a question in English, writes SQL
against the four Lakehouse tables, runs it, and explains the result. Wrapping it
as a tool is a matter of starting a conversation and reading back both halves of
the reply: the prose, and the attachment holding the generated SQL with its rows.

Both halves go into the finding. The prose is the answer, and the SQL is how you
check the answer, which is worth keeping when the next tool's choice depends on
this one's numbers.

This is the only tool that can see a sensor reading. Your graph has `Sensor`
nodes but no readings on them: 155,000 timestamped values live in Delta, where
scanning them is cheap. Remember that when you read the routing prompt.

In [ ]:
genie_node = build_genie_node(GENIE_SPACE_ID, workspace)

# One call, on its own, before any routing is involved.
probe = genie_node({"question": "What is the average EGT for aircraft N10000?"})
print(probe["findings"][-1]["content"][:1200])

## Section 4: `cypher_node`, the graph tool

Text to Cypher, against your own Aura instance over Bolt. The LLM gets the schema
and the question, writes a query, and the query runs.

Three details do the work:

**The schema is the one your graph actually has.** `tools.GRAPH_SCHEMA` lists
what Lab 2 loaded plus what Lab 3 added, and nothing else. It has no `Reading`
label, because your graph has no `Reading` label. It spells the system type
`Hydraulics`, plural, because that is the string in the data, and it says there
are only three system types so the model does not invent a fuel system to match a
question about fuel. Every one of those details was measured rather than
remembered. A schema that promises something the graph does not have produces
Cypher that runs cleanly, returns zero rows, and leaves an agent politely
reporting that it found nothing.

**It runs in a read transaction.** Aura rejects a write inside one, so a
generated `MERGE` fails at the server. There is a regular expression check as
well, ahead of it, so the refusal is a sentence rather than a driver error. Two
guards, because the first one is a regular expression.

**A failed query gets one retry with its error attached.** Most text to Cypher
failures are a mistyped property or a relationship pointing the wrong way, and
the error message says which.

In [ ]:
cypher_node = build_cypher_node(driver, llm, database=NEO4J_DATABASE)

probe = cypher_node(
    {"question": "Which aircraft have had critical maintenance events on their engine systems, and which components were involved?"}
)
print(probe["findings"][-1]["content"][:1500])

Three things about `OperatingLimit` are worth reading before you write your own
questions.

**A name can appear twice.** Lab 2 loads twenty canonical limits from CSV, four
per aircraft model, each carrying a `limit_id`. Lab 3 notebook 01 then extracts
limits from the manual prose under the same `<parameterName> - <aircraftType>`
name, and those carry no `limit_id`. So `EGT - A320-200` can exist twice with
different bounds, and a question about the documented limit needs
`WHERE ol.limit_id IS NOT NULL` to pick the canonical row. That is the lesson:
extraction and ingestion can collide on an identifier neither one owns.

**Ten of the twenty have no floor.** `minValue` is null on the Vibration and
N1Speed limits for each of the five models, because those are ceilings and
nothing else. The schema says to check `IS NOT NULL` before comparing, or the
row drops out with no error. Both bounds are Double, so nothing needs casting.

**Every limit belongs to a regime.** Each one carries the phase of flight it
applies to. A takeoff bound held against cruise readings is a category error
rather than a comparison, and it reports the whole fleet out of range.

## Section 5: `graphrag_node`, the manual tool

This is the Lab 3 notebook 02 retriever, wrapped as a node.

The question is embedded with `databricks-bge-large-en`, the vector index returns
the closest manual chunks, and then a Cypher tail runs from each hit. The tail is
the whole point. Without it this is vector search with extra steps.

The tail walks two ways from the matched chunk:

```cypher
WITH node
OPTIONAL MATCH (previous:Chunk)-[:NEXT_CHUNK]->(node)
OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(following:Chunk)
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (doc)-[:APPLIES_TO]->(a:Aircraft)-[:HAS_SYSTEM]->(s:System)
```

Sideways along `NEXT_CHUNK`, so a procedure that got split across a chunk
boundary arrives in one piece. Upward through the `Document` to the aircraft the
manual applies to and that aircraft's systems, so the answer knows which tail
numbers it is about. Neither of those is in the embedding. Both are one hop away
in the graph.

**When the index is missing.** `VectorCypherRetriever` raises if it cannot find
its index, which would turn a skipped Lab 3 into a failure in this cell, several
cells before the agent exists. So the builder checks first and returns a node
that explains itself instead. The agent still builds and still runs, with two
tools rather than three.

In [ ]:
graphrag_node = build_graphrag_node(
    driver, llm, embedder, database=NEO4J_DATABASE, top_k=3
)

if getattr(graphrag_node, "available", False):
    probe = graphrag_node(
        {"question": "What does the manual say about EGT exceedance during takeoff?"}
    )
    print(probe["findings"][-1]["content"][:1500])
else:
    print("Manual tool unavailable. This is what the supervisor would receive:\n")
    print(graphrag_node({"question": "anything"})["findings"][-1]["content"])

## Section 6: The supervisor prompt

Lab 4 Part B wrote routing instructions for two agents and let Agent Bricks
handle the rest. The same instructions, with a third tool added, is the starting
point here. What has to change is the part that separates `cypher_node` from
`graphrag_node`.

Both of them end in a Neo4j traversal. A model that describes the tools by what
they do at the end cannot tell them apart, and it will send manual questions to
Cypher, where they return nothing, because the manual text is not a property it
can filter on.

So the prompt tells the model to decide on where the question **starts**:

> Starts with a name you could put in a `WHERE` clause -> `cypher_node`
> Starts with a phrase you would search a manual for -> `graphrag_node`

With the pairs that make the line concrete:

| Question | Route | Why |
|---|---|---|
| "What maintenance events did N10004 have?" | `cypher_node` | N10004 is a node |
| "What is the procedure for an EGT exceedance?" | `graphrag_node` | That is a phrase in a manual, not a node |
| "What is the documented EGT limit for the A320-200?" | `cypher_node` | A `maxValue` property on an `OperatingLimit` |
| "How do I troubleshoot engine vibration?" | `graphrag_node` | A procedure, so it lives in the manual text |

The prompt also carries four rules about when to stop. A supervisor with a loop
back to itself will call the same tool three times if nothing tells it not to,
because calling a tool again always looks safer than answering. The rules say to
call each tool at most once and to synthesize as soon as every part of the
question has something against it. The `MAX_TOOL_CALLS` budget is the backstop
under those rules, not a substitute for them.

Print `SUPERVISOR_PROMPT` from `tools.py` and read it. It is the part of this lab
you are most likely to change for your own domain.

In [ ]:
from tools import SUPERVISOR_PROMPT

# Drop graphrag_node from the supervisor's options when the index is absent, so
# it stops offering an answer it cannot produce.
AVAILABLE_TOOLS = ["genie_node", "cypher_node"]
if getattr(graphrag_node, "available", False):
    AVAILABLE_TOOLS.append("graphrag_node")

supervisor_node = build_supervisor_node(llm, available_tools=AVAILABLE_TOOLS)
synthesize_node = build_synthesize_node(llm)

print(f"Tools the supervisor can call: {', '.join(AVAILABLE_TOOLS)}\n")
print(SUPERVISOR_PROMPT.split("## The question")[0])

## Section 7: Wiring the graph

Five nodes and one decision. `START` goes to the supervisor. The supervisor's
`route` picks a tool or picks `synthesize`. Every tool goes back to the
supervisor. `synthesize` goes to `END`.

The edge back from each tool is what makes this a supervisor rather than a
router. A router picks one tool and answers. This one sees what the tool
returned, and gets to pick again.

State is a `TypedDict` with five keys. `trace` is the list of tools called in
order, which is the record Section 9 measures.

In [ ]:
from langgraph.graph import END, START, StateGraph

from tools import MAX_TOOL_CALLS, AgentState, route_from_supervisor

builder = StateGraph(AgentState)

builder.add_node("supervisor", supervisor_node)
builder.add_node("genie_node", genie_node)
builder.add_node("cypher_node", cypher_node)
builder.add_node("graphrag_node", graphrag_node)
builder.add_node("synthesize", synthesize_node)

builder.add_edge(START, "supervisor")

# The one decision in the graph. route_from_supervisor reads state["route"] and
# returns the name of the next node.
builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {
        "genie_node": "genie_node",
        "cypher_node": "cypher_node",
        "graphrag_node": "graphrag_node",
        "synthesize": "synthesize",
    },
)

# Every tool reports back rather than answering.
for tool_name in ("genie_node", "cypher_node", "graphrag_node"):
    builder.add_edge(tool_name, "supervisor")

builder.add_edge("synthesize", END)

agent = builder.compile()
print(f"Agent compiled. Tool call budget per question: {MAX_TOOL_CALLS}")

In [ ]:
from typing import Any


def ask(question: str, *, verbose: bool = True) -> dict[str, Any]:
    """Run one question through the agent and show the route it took."""
    result = agent.invoke({"question": question, "trace": [], "findings": []})
    if verbose:
        print(f"Q: {question}")
        print(f"Route: {' -> '.join(result['trace']) or '(no tool called)'}\n")
        print(result["answer"])
        print("-" * 78)
    return result

## Section 8: The four routing cases

Three questions that should each land on exactly one tool, then one that needs
all three.

In [ ]:
case_1 = ask("What is the average EGT for aircraft N10000 in August 2024?")

In [ ]:
case_2 = ask("Which aircraft have had critical maintenance events on their engine systems, and which components were involved?")

In [ ]:
case_3 = ask("What does the maintenance manual say about EGT exceedance procedures?")

### The anchor question

One question, three stores. The readings that say which engine is running hot are
in Delta. The maintenance history for that engine is in the graph. The procedure
is in a manual chunk. No single tool answers it, and the supervisor has to work
that out from the question rather than from a rule you wrote.

Watch the route. Three tools in sequence is the point of the lab.

In [ ]:
ANCHOR_QUESTION = (
    "Which engines are showing abnormal EGT readings, what maintenance history "
    "do those aircraft have, and what does the maintenance manual say to do "
    "about high EGT?"
)

anchor = ask(ANCHOR_QUESTION)

## Section 9: Measuring the routing

Reading three good answers proves the tools work. It does not prove the routing
works, because three questions is not a measurement.

The set below is twelve questions, four per tool, run through the supervisor
alone with no findings to reason from. That is the routing decision in isolation:
what the model picks from the question text alone, on the first call.

The `cypher_node` against `graphrag_node` number is reported on its own. Folding
it into an overall score hides it, and that pair is where misrouting shows up
first, because `VectorCypherRetriever` makes the two tools genuinely adjacent.

In [ ]:
ROUTING_CASES = [
    # genie_node: a measured value, or an aggregate over measured values
    ("What is the average EGT for aircraft N10000 in August 2024?", "genie_node"),
    ("Compare average vibration readings between B737-800 and A320-200 aircraft.", "genie_node"),
    ("What was the maximum fuel flow recorded in August 2024?", "genie_node"),
    ("Show the daily trend of N1 speed for aircraft N10000.", "genie_node"),
    # cypher_node: starts from a named entity, answered by following relationships
    ("Which aircraft have had critical maintenance events on their engine systems, and which components were involved?", "cypher_node"),
    ("What maintenance events has aircraft N10004 had, and how severe were they?", "cypher_node"),
    ("Which components are in the hydraulic system of aircraft N10000?", "cypher_node"),
    ("Which aircraft have had the most part removals, and for what reason?", "cypher_node"),
    # graphrag_node: starts from language in a manual
    ("What does the maintenance manual say about EGT exceedance procedures?", "graphrag_node"),
    ("How do I troubleshoot excessive engine vibration?", "graphrag_node"),
    ("What is the documented inspection procedure after a hard landing?", "graphrag_node"),
    ("What steps does the manual give for a hydraulic pressure loss?", "graphrag_node"),
]

results = []
skipped = []
for question, expected in ROUTING_CASES:
    if expected not in AVAILABLE_TOOLS:
        # Scoring a tool the supervisor was never offered measures nothing.
        skipped.append((question, expected))
        continue
    chosen = supervisor_node({"question": question, "trace": [], "findings": []})["route"]
    results.append((question, expected, chosen))
    mark = "PASS" if chosen == expected else "FAIL"
    print(f"{mark}  expected {expected:<14} got {chosen:<14} {question[:52]}")

for question, expected in skipped:
    print(f"SKIP  {expected} is unavailable in this workspace: {question[:52]}")

In [ ]:
def accuracy(rows) -> str:
    if not rows:
        return "n/a"
    hits = sum(1 for _, expected, chosen in rows if expected == chosen)
    return f"{hits}/{len(rows)} ({100 * hits / len(rows):.0f}%)"


graph_pair = [row for row in results if row[1] in ("cypher_node", "graphrag_node")]

print(f"Overall routing accuracy:              {accuracy(results)}")
print()
print(f"  genie_node questions:                {accuracy([r for r in results if r[1] == 'genie_node'])}")
print(f"  cypher_node questions:               {accuracy([r for r in results if r[1] == 'cypher_node'])}")
print(f"  graphrag_node questions:             {accuracy([r for r in results if r[1] == 'graphrag_node'])}")
print()
print(f"cypher_node vs graphrag_node, on its own: {accuracy(graph_pair)}")
print("  This is the number that matters. Both tools end in a traversal, so this")
print("  pair is where a weak routing prompt fails first.")
if skipped:
    print(f"\n{len(skipped)} question(s) skipped because their tool is unavailable here.")

Anything that failed is a prompt problem, not a model problem. Take the question
that went wrong, work out which sentence in `SUPERVISOR_PROMPT` should have
caught it, and add the pair to the examples in the "line between them" section.
Rerun the cell. Routing prompts are tuned by looking at what they got wrong, the
same way you would tune a classifier.

If the pair number stays low after a few rounds, the other lever is
`SUPERVISOR_MODEL` in `tools.py`. It is one named constant, declared once, so
swapping to a stronger tool-calling endpoint is a one-line change and nothing
else in the lab moves.

## Section 10: Optional, hybrid retrieval

Skip this unless you ran [Lab 3 notebook
03](../Lab_3_Semantic_Search/03_hybrid_retrievers.ipynb).

`graphrag_node` finds manual passages by meaning, which is what you want for
"what do I do about high exhaust temperature" when the manual says "EGT
exceedance". It is the wrong instrument for an exact string. A part number, a
fault code, a specific engine designation: those either appear in the text or
they do not, and an embedding of `CFM56-7B` is close to an embedding of every
other engine model.

`HybridCypherRetriever` runs the vector index and the `maintenanceChunkText`
fulltext index together and merges the rankings. Same Cypher tail, so the graph
context is unchanged. Build a second node from it and add it to the graph the
same way, or swap it in for the retriever inside `build_graphrag_node`.

In [ ]:
from tools import FULLTEXT_INDEX_NAME, MANUAL_CONTEXT_QUERY, format_manual_chunk

records, _, _ = driver.execute_query(
    "SHOW INDEXES YIELD name, type, state "
    "WHERE name = $name AND type = 'FULLTEXT' AND state = 'ONLINE' "
    "RETURN count(*) AS found",
    name=FULLTEXT_INDEX_NAME,
    database_=NEO4J_DATABASE,
)

if records[0]["found"] and getattr(graphrag_node, "available", False):
    from neo4j_graphrag.generation import GraphRAG
    from neo4j_graphrag.retrievers import HybridCypherRetriever

    hybrid_retriever = HybridCypherRetriever(
        driver=driver,
        neo4j_database=NEO4J_DATABASE,
        vector_index_name=VECTOR_INDEX_NAME,
        fulltext_index_name=FULLTEXT_INDEX_NAME,
        retrieval_query=MANUAL_CONTEXT_QUERY,
        embedder=embedder,
        result_formatter=format_manual_chunk,
    )
    hybrid_rag = GraphRAG(llm=llm, retriever=hybrid_retriever)

    answer = hybrid_rag.search(
        "What are the maintenance requirements for the CFM56-7B engine?",
        retriever_config={"top_k": 3},
    )
    print(answer.answer)
else:
    print(
        f"Fulltext index '{FULLTEXT_INDEX_NAME}' not found. "
        "Run Lab 3 notebook 03 to build it, then rerun this cell."
    )

## Section 11: Try your own

Ask it something the workshop did not plan for. Watch the route before you read
the answer, because a wrong answer with a sensible route is a tool problem and a
wrong answer with a strange route is a prompt problem, and those are fixed in
different files.

Questions worth trying:

- "Which aircraft has the highest vibration, and has it had a maintenance event?"
- "What is the documented N1 speed limit for the B737-800, and how do current readings compare?"
- "Which components on the hydraulics systems have failed, and what does the manual say to check?"

In [ ]:
my_question = "Which aircraft has the highest average vibration, and has it had any maintenance events?"

my_result = ask(my_question)

In [ ]:
driver.close()
print("Neo4j connection closed.")

## What you built

A supervisor over three stores, with the routing rule that matters written down
rather than assumed. The three tools are the three labs before this one, and the
prompt in `tools.py` is the only new idea.

Notebook 02 takes the same graph, logs it as an MLflow model, deploys it to Model
Serving, and evaluates it against a question set with MLflow's LLM judges. Lab 6
gives it memory, in Neo4j, so it can be asked a follow-up.